## Example 1: A slash command handled without a network

In [ ]:
import hashlib
import hmac
import importlib.metadata
import time

from slack_bolt import App, BoltRequest
from slack_bolt.authorization import AuthorizeResult

# Example 1: dispatch a signed /deploy request into an app with a local authorize(); ack() becomes the HTTP response
assert importlib.metadata.version("slack-bolt") == "1.30.0"
app = App(signing_secret="secret", authorize=lambda **kw: AuthorizeResult(enterprise_id=None, team_id="T1", bot_token="xoxb-test", bot_user_id="U0"))


@app.command("/deploy")
def deploy(ack, command):
    ack(f"Deploying {command['text']}")


body = "command=%2Fdeploy&text=prod&user_id=U1&team_id=T1"
ts = str(int(time.time()))
sig = "v0=" + hmac.new(b"secret", f"v0:{ts}:{body}".encode(), hashlib.sha256).hexdigest()
response = app.dispatch(BoltRequest(body=body, headers={"x-slack-request-timestamp": [ts], "x-slack-signature": [sig]}))
assert response.status == 200 and "Deploying prod" in response.body, (response.status, response.body)
print("Example 1 OK:", response.body)

## Example 2: A bad signature is refused

In [ ]:
import time

from slack_bolt import App, BoltRequest

# Example 2: the request-verification middleware answers 401 before any listener runs
app = App(signing_secret="secret", authorize=lambda **kw: None)
called = []
app.command("/deploy")(lambda ack: called.append(1) or ack())
headers = {"x-slack-request-timestamp": [str(int(time.time()))], "x-slack-signature": ["v0=not-a-signature"]}
response = app.dispatch(BoltRequest(body="command=%2Fdeploy&text=x", headers=headers))
assert response.status == 401 and not called, (response.status, called)
print("Example 2 OK:", response.status)

## Example 3: Events API: an app_mention event reaches its listener

In [ ]:
import hashlib
import hmac
import json
import time

from slack_bolt import App, BoltRequest
from slack_bolt.authorization import AuthorizeResult

# Example 3: a JSON event callback; with process_before_response the listener runs before the reply
app = App(signing_secret="secret", process_before_response=True,
          authorize=lambda **kw: AuthorizeResult(enterprise_id=None, team_id="T1", bot_token="xoxb-test", bot_user_id="U0"))
seen = []
app.event("app_mention")(lambda event: seen.append(event["text"]))
body = json.dumps({"type": "event_callback", "team_id": "T1", "api_app_id": "A1",
                   "event": {"type": "app_mention", "user": "U1", "text": "<@U0> hi", "channel": "C1", "ts": "1.0"}})
ts = str(int(time.time())); sig = "v0=" + hmac.new(b"secret", f"v0:{ts}:{body}".encode(), hashlib.sha256).hexdigest()
response = app.dispatch(BoltRequest(body=body, headers={"content-type": ["application/json"], "x-slack-request-timestamp": [ts], "x-slack-signature": [sig]}))
assert response.status == 200 and seen == ["<@U0> hi"], (response.status, seen)
print("Example 3 OK:", seen)

## Example 4: Interactive actions: a button click

In [ ]:
import hashlib
import hmac
import json
import time
import urllib.parse

from slack_bolt import App, BoltRequest
from slack_bolt.authorization import AuthorizeResult

# Example 4: block_actions arrive as a form field holding JSON; the action listener acks them
app = App(signing_secret="secret", process_before_response=True,
          authorize=lambda **kw: AuthorizeResult(enterprise_id=None, team_id="T1", bot_token="xoxb-test", bot_user_id="U0"))
clicks = []
@app.action("rerun")
def rerun(ack, action):
    ack(); clicks.append(action["value"])
payload = {"type": "block_actions", "team": {"id": "T1"}, "user": {"id": "U1"}, "api_app_id": "A1",
           "actions": [{"type": "button", "action_id": "rerun", "block_id": "b", "value": "42", "action_ts": "1"}]}
body = "payload=" + urllib.parse.quote(json.dumps(payload))
ts = str(int(time.time())); sig = "v0=" + hmac.new(b"secret", f"v0:{ts}:{body}".encode(), hashlib.sha256).hexdigest()
response = app.dispatch(BoltRequest(body=body, headers={"x-slack-request-timestamp": [ts], "x-slack-signature": [sig]}))
assert response.status == 200 and clicks == ["42"], (response.status, clicks)
print("Example 4 OK:", clicks)

## Example 5: The asyncio app

In [ ]:
import hashlib
import hmac
import time

from slack_bolt.async_app import AsyncApp
from slack_bolt.authorization import AuthorizeResult
from slack_bolt.request.async_request import AsyncBoltRequest

# Example 5: AsyncApp runs async listeners; async_dispatch returns the response
async def authorize(**kwargs):
    return AuthorizeResult(enterprise_id=None, team_id="T1", bot_token="xoxb-test", bot_user_id="U0")
app = AsyncApp(signing_secret="secret", authorize=authorize)
@app.command("/ping")
async def ping(ack):
    await ack("pong")
body = "command=%2Fping&text=&user_id=U1&team_id=T1"
ts = str(int(time.time())); sig = "v0=" + hmac.new(b"secret", f"v0:{ts}:{body}".encode(), hashlib.sha256).hexdigest()
response = await app.async_dispatch(AsyncBoltRequest(body=body, headers={"x-slack-request-timestamp": [ts], "x-slack-signature": [sig]}))
assert response.status == 200 and "pong" in response.body, (response.status, response.body)
print("Example 5 OK:", response.body)